## Computational Aspects of Complex Reflection Groups

Götz Pfeiffer - University of Galway

# 4. Vectors: Enumerating Modules and Hecke Algebras

![Benches](images/benches.jpg)

## Setup

In [ ]:
using LinearAlgebra
using OrbitAl
using OrbitAl.permutation, OrbitAl.orbits
using Graphs, GraphPlot

All orbit algorithms from Parts 1–3 are available via `OrbitAl`.

In [ ]:
# Algorithms from previous parts are available via OrbitAl

##  Spinning: $K$-Linear Orbit Algorithm

* In the **linear** version, the points of an orbit are linearly independent vectors in $K^n$.
* And the acting operators are like matrices in $K^{n\times n}$.

In [ ]:
function inSpan(list, z)
    isempty(list) && return false
    A = hcat(list...)
    return rank(A) == rank(hcat(A, z))
end

function spinning(aaa, x, under)
    list = [x]
    for y in list, a in aaa
        z = under(y, a)
        inSpan(list, z) || push!(list, z)
    end
    return list
end

### Example

In [ ]:
# companion matrix of x^5 + x^4 + x^3 + x^2 + x + 1
# column vector convention: action is v ↦ A*v
A = Rational{Int}[
    0  0  0  0 -1;
    1  0  0  0 -1;
    0  1  0  0 -1;
    0  0  1  0 -1;
    0  0  0  1 -1
]

In [ ]:
spinning([A], Rational{Int}[0,0,1,0,0], (v,M) -> M*v)

In [ ]:
spinning([A], Rational{Int}[1,1,0,1,1], (v,M) -> M*v)

In [ ]:
spinning([A], Rational{Int}[1,0,1,0,1], (v,M) -> M*v)

##  Spinning with Images

* We will need to be able to convert between sparse and dense representations of vectors.

In [ ]:
function sparseVec(vec)
    pos = findall(!iszero, vec)
    return (pos = pos, val = vec[pos])
end

function denseVec(l, v)
    vec = zeros(eltype(v.val), l)
    vec[v.pos] = v.val
    return vec
end

In [ ]:
vec = [0, 0, 1, 1, 0]
v = sparseVec(vec)

In [ ]:
denseVec(length(vec), v) == vec

* spinning with images (cf. orbit with images)

* In the linear version the images are vectors: linear combination of the base points.

In [ ]:
function solutionMat(list, z)
    isempty(list) && return nothing
    k, n = length(list), length(z)
    aug = hcat(list..., z)           # n × (k+1) augmented matrix [A | z]
    r, pivots = 0, Int[]
    for col in 1:k
        pivot = findfirst(!iszero, aug[r+1:end, col])
        isnothing(pivot) && continue
        r += 1;  pivot += r - 1
        aug[[r, pivot], :] = aug[[pivot, r], :]
        aug[r, :] ./= aug[r, col]    # normalize pivot row (exact for Rational)
        for row in [1:r-1; r+1:n]
            iszero(aug[row, col]) && continue
            aug[row, :] -= aug[row, col] .* aug[r, :]
        end
        push!(pivots, col)
    end
    any(!iszero, aug[r+1:end, end]) && return nothing   # z not in span
    c = zeros(eltype(z), k)
    for (i, col) in enumerate(pivots)
        c[col] = aug[i, end]
    end
    return c
end

function spinning_with_images(aaa, x, under)
    list = [x]
    images = [[] for _ in aaa]
    for (i, y) in enumerate(list)
        for (k, a) in enumerate(aaa)
            z = under(y, a)
            v = solutionMat(list, z)
            if isnothing(v)
                push!(list, z)
                v = (pos=[length(list)], val=[one(eltype(z))])
            else
                v = sparseVec(v)
            end
            push!(images[k], v)
        end
    end
    return (list = list, images = images)
end

### Example: Specht-Module

* Everybody knows that Specht modules are made of Standard Young Tableaus ...
* Here is a random one, of shape $\lambda = (2,2,1)$ (for the rest see the exercise on SYTs in part 1).

In [ ]:
tab = [[1,4], [2,5], [3]]

* and the corresponding symmetric group

In [ ]:
N = sum(length(row) for row in tab)
gens = transpositions(N)

* each tableau (standard or not) yields a (row) word, recording for each number the row it is in.

In [ ]:
function rowWord(tab)
    word = zeros(Int, sum(length(row) for row in tab))
    for (i, row) in enumerate(tab)
        word[row] .= i
    end
    return word
end

function colWord(tab)
    word = zeros(Int, sum(length(row) for row in tab))
    for row in tab, (c, elem) in enumerate(row)
        word[elem] = c
    end
    return word
end

In [ ]:
rowWord(tab)

* the column stabilizer is the stabilizer of the column word.
* the orbit of a tableau under the column stabilizer is a (column) tabloid.

In [ ]:
cword = colWord(tab)
stab_result = orbit_with_stabilizer(gens, cword, permuted)
colStab = filter(!isidentity, stab_result.stab)

onTab(t, g) = [[x^g for x in row] for row in t]
tabs = orbit_with_transversal(colStab, tab, onTab)
length(tabs.list)

* the symmetric group acts on the row words

In [ ]:
orb = orbit_with_images(gens, rowWord(tab), permuted)
words = orb.list
perms = [Perm(img) for img in orb.images]
length(words)

* the polytabloid is a signed combination of tabloids, written as a vector in the permutation module.

In [ ]:
sign_perm(g) = prod((-1)^(length(c) - 1) for c in cycles(g); init=1)

vec = zeros(Int, length(words))
for (i, t) in enumerate(tabs.list)
    pos = findfirst(==(rowWord(t)), words)
    vec[pos] = sign_perm(tabs.reps[i])
end
vec

* apply spinning to our polytabloid

In [ ]:
vvv = spinning_with_images(perms, Rational{Int}.(vec), permuted)
length(vvv.list)  # dimension of the Specht module

* How to convert a list of sparse vectors into a matrix (cf. `PermList`)

In [ ]:
function mat_list(images, n)
    T = isempty(images) ? Rational{Int} : eltype(images[1].val)
    M = zeros(T, length(images), n)
    for (i, v) in enumerate(images)
        M[i, :] = denseVec(n, v)
    end
    return M
end

In [ ]:
mats = [mat_list(vvv.images[k], length(vvv.list)) for k in eachindex(vvv.images)]

In [ ]:
for m in mats
    display(m)
end

## Schreier Matrices and the Group Algebra

* Another way to produce matrices from an orbit is to read the resulting permutations as **permutation matrices**.

In [ ]:
gens4 = transpositions(4)
orb4 = orbit_with_images(gens4, 4, onPoints)

function permMat(img)
    n = length(img)
    M = zeros(Int, n, n)
    for (i, j) in enumerate(img)
        M[i, j] = 1
    end
    return M
end

pm = [permMat(img) for img in orb4.images]

In [ ]:
for m in pm
    display(m)
end

* Next, we use the Schreier generators as entries in those matrices, rather than just $1$.
* Once again, only a small modification of an earlier version of the orbit algorithm, `orbit_with_images`, is needed.
* Storing these images as **sparse vectors** will allow to use `mat_list` to recover the matrices.

In [ ]:
function orbit_with_schreier(aaa, x, under)
    list = [x]
    reps = [aaa[1]^0]
    images = [[] for _ in aaa]
    for (i, y) in enumerate(list)
        for (k, a) in enumerate(aaa)
            z = under(y, a)
            l = findfirst(==(z), list)
            if isnothing(l)
                push!(list, z)
                push!(reps, reps[i] * a)
                l = length(list)
            end
            push!(images[k], (pos=[l], val=[reps[i] * a / reps[l]]))
        end
    end
    return (list = list, images = images)
end

In [ ]:
orb_s = orbit_with_schreier(gens4, 4, onPoints)

In [ ]:
ncycles(g) = filter(c -> length(c) > 1, cycles(g))  # non-trivial cycles only

for (k, imgs) in enumerate(orb_s.images)
    println("Generator $k  $(ncycles(gens4[k])):")
    for (i, v) in enumerate(imgs)
        g = v.val[1]
        println("  coset $i → coset $(v.pos[1]),  g = ", isidentity(g) ? "id" : ncycles(g))
    end
end

* These matrices form a representation of the group algebra $\mathbb{C}G$, acting regularly on $\mathbb{C}G$ regarded as a right $\mathbb{C}G$-module over the group algebra $\mathbb{C}H$ of the stabilizer $H$:
$$
  \mathbb{C}G = \bigoplus_{t \in T} \mathbb{C}H t,
$$
where $T$ is the transversal.

* Let's call the resulting matrices the **Schreier matrices** of the action.
* Replacing each Schreier generator $h \in H$ (a group element) by its image under a matrix representation of $H$ yields an **induced representation**.  Substituting the identity gives back the permutation representation above.

In [ ]:
# Verify the permutation matrices (pm) satisfy S_4 Coxeter relations
I4 = Matrix{Int}(I, 4, 4)
involutions = all(pm[k]^2 == I4 for k in eachindex(pm))
braid = pm[1]*pm[2]*pm[1] == pm[2]*pm[1]*pm[2]
commute = pm[1]*pm[3] == pm[3]*pm[1]  # generators 2 apart commute
(involutions = involutions, braid = braid, commute = commute)

In [ ]:
# The Schreier generator is the identity when the coset moves forward in the orbit tree,
# and a non-trivial group element when it cycles back.
# Replacing each Schreier generator by its image under a representation of the stabilizer
# yields an induced representation — here with the trivial rep we recover the permutation rep.
pm[1] * pm[2]

* The Schreier matrices can serve as a blueprint for induced representations: Take any matrix representation of $H$ and replace the Schreier generators $h \in H$ in the Schreier matrix of $g \in G$ by the matrices representing $h$:  the result (with a suitable interpretation of $0$) will be a matrix representing $g$. Replacing all Schreier generators by the trivial representation $1$ yields the above  permutation representation of $G$ on the cosets of $H$.

* In the case of a Coxeter group acting on the cosets of a parabolic subgroup, the Schreier generators are either trivial, or simple reflections, thanks to the following result.

<div class="alert alert-danger">

**Theorem** (Deodhar's Lemma) **.**  Let $(W, S)$ be a finite Coxeter group, and let $J \subseteq S$.
    
* Let $x \in X_J$ and $s \in S$.  Then either $xs \in X_J$ or $xs = ux$ for some $u \in J$.
    
</div>

* Note how $u = xsx^{-1}$ is a Schreier generator if $xs \notin X_J$.

##  Iwahori-Hecke Algebra

* The Iwahori-Hecke algebra $H$ of $(W, S)$ is ...

<div class="alert alert-danger">

**Theorem** (Deodhar's Lemma for Iwahori-Hecke algebras) **.**  Let $(W, S)$ be a finite Coxeter group, and the $J \subseteq S$.
    
* Let $x \in X_J$ and $s \in S$.  Then either 
$$
T_x T_s = \begin{cases}
(q-1)T_x + qT_{xs}, & \text{if } \ell(xs) < \ell(x),\\
T_{xs}, & \text{if } \ell(xs) > \ell(x) \text { and } xs \in X_J,\\
T_u T_x, & \text{if } \ell(xs) > \ell(x) \text { and } xs \notin X_J,
\end{cases}
$$
for some $u \in J$. 
</div>

* We thus get Schreier matrices for the generators $T_s$ of $H$.  For example: ...
$$
T_1 \mapsto  \left[ \begin{array}{cccc}
      T_{1}&.&.&.\\ .&T_{1}&.&.\\ .&.&.&1\\ .&.&q&q{-}1
    \end{array} \right],
    \quad
T_2 \mapsto  \left[ \begin{array}{cccc}
      T_{2}&.&.&.\\ .&.&1&.\\ .&q&q{-}1&\\ .&.&.&T_{1}
    \end{array} \right],
    \quad
T_3 \mapsto  \left[ \begin{array}{cccc}
      .&1&.&.\\ q&q{-}1&.&.\\ .&.&T_{2}&.\\ .&.&.&T_{2}
    \end{array} \right]
$$

* As coset table, aka image list of sparse vectors.
$$
  \begin{array}{l|ccc}
    x &x.1&x.2&x.3\\\hline
    x_1 = \emptyset & 1 \cdot x & 2 \cdot x & \underline{x_2}\\
    x_2 = 3 & 1 \cdot x & \underline{x_3} & (q{-}1)x + qx_1 \\
    x_3 = 32 & \underline{x_4} & (q{-}1) x + qx_2 & 2 \cdot x \\
    x_4 = 321 & (q{-}1)x + q x_3 & 1 \cdot x & 2 \cdot x
  \end{array}
$$

* Can we find such a coset table for the Hecke algebra of a complex reflection group? Yes ...

##  Linear Coset Enumeration

* Like the spinning algorithm is a linear version of the orbit algorithm, there is a lineasr version of the coset enumeration procedure.  Naturally, at certain stages of the procedure, a linear result has to be expected, and handled.

* ... details omitted ...

* Example $G(3,3,3)$.

In [ ]:
# G333 presentation needed here — defined in enumerate.ipynb
# G = G333

In [ ]:
# data = coset_table(G, [])

In [ ]:
# nodes = filter(is_active, data)

In [ ]:
# gens_g333 = [Perm([flat(node.next[i]).idx for node in nodes]) for i in eachindex(G.gens)]

In [ ]:
# sizeOfGroup(PermGp(gens_g333, gens_g333[1]^0))

In [ ]:
# edges = union([[(node.idx, flat(node.next[i]).idx) for node in nodes] for i in eachindex(G.gens)]...)
# gplot(SimpleDiGraph(Edge.(filter(e -> e[1] != e[2], edges))))

$$
  \begin{array}{l|cccc}
    x &x.t_0&x.t_1&x.t_2&x.s_3\\\hline
    x_0 =  & t_0 \cdot x & t_1 \cdot x & t_2 \cdot x & \underline{x_1} \\
    x_1 = s_3 & \underline{x_2} & \underline{x_3} & \underline{x_4} & (q{-}1) x + q x_0\\
    x_2 = s_3 t_0 & (q{-}1) x + q x_1 & \eqref{2.1} & x_5 & t_0 \cdot x_2 \\
    x_3 = s_3 t_1 & \underline{x_5} & (q{-}1) x + q x_1 & \eqref{3.2} & t_1 \cdot x_3 \\
    x_4 = s_3 t_2 & \underline{x_6} & x_5 & (q{-}1) x + q x_1 & t_2 \cdot x_4 \\
    x_5 = s_3 t_1 t_0 & (q{-}1) x + q x_3 & (q{-}1) x + q x_4 & (q{-}1) x + q x_2 & \underline{x_7} \\
    x_6 = s_3 t_2 t_0 & (q{-}1) x + q x_4 & \eqref{6.1} & (q{-}1) x_5 + q x_3 & \underline{x_8} \\
    x_7 = s_3 t_1 t_0 s_3 & t_1 \cdot x & t_2 \cdot x & t_0 \cdot x & (q{-}1) x + q x_5 \\
    x_8 = s_3 t_2 t_0 s_3 & t_2 \cdot x & \eqref{8.1} & \eqref{8.2} & (q{-}1) x + q x_6\\
  \end{array}
$$

* where
  * $x_2.t_1 = (q{-}1) x_3 + (1{-}q) x_4 + x_6$
  * $x_3.t_2 = (1{-}q) x_2 + (q{-}1) x_3 + x_6$
  * $  x_6.t_1 
  = (q{-}1) x_5 + (1{-}q)(q{-}1) x_4 + (q{-}1) x
  + (1{-}q) q x_1 + q x_2$
  * $  x_8.t_1 =
  (q{-}1)t_2\cdot x_5
  + q(q{-}1)t_2t_0' \cdot x_3
  +  t_0 \cdot x_8
  + (1{-}q)t_0t_2 \cdot x_4
  + q(1{-}q)t_1 \cdot x_1$
  * $  x_8.t_2 =
   q(q{-}1) x_3
   + (q{-}1)t_0 \cdot x_5
   + t_1 \cdot x_8
   + (1{-}q)t_1 \cdot x_6
   + q(1{-}q)t_1t_0' \cdot x_2$

## Exercises, etc.

* Expand `spinning` into `spinning_with_matrices`: instead of tracking a list of basis vectors, directly build the representation matrices as the orbit grows.
* Try other shapes: compute the Specht module of shape $\lambda = (3,2)$ and verify its dimension is $5$.
* Verify that the matrices in `mats` satisfy the Coxeter relations of $S_5$ (involutions and braid relations).
* Implement a Hecke algebra version of `spinning_with_images` where the coefficients in `solutionMat` come from a polynomial ring $\mathbb{Z}[q]$.

* ...